In [1]:
import manim as mn
from manim import *
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

config.media_width = "75%"
config.verbosity = "WARNING"

print(mn.__version__)

0.20.1


In [2]:
%%manim -qm DopplerEffect

current_frame = 0
class DopplerEffect(Scene):
    
    def construct(self):
        
        source = Dot(point=LEFT*2.5)
        waves = VGroup()

        observer = Dot(point=DOWN*0.5, color=GREEN)

        
        def addwavefront(mob,dt):
            global current_frame
            wavefront=Circle(
                radius=0.01,
                color=YELLOW,
                stroke_width=1.5,
                arc_center=source.get_center()
            ) 
            current_frame += 1
            if current_frame%8 == 0:
                waves.add(wavefront)
            return current_frame

        def expandwaves(mob, dt):
            for wavefront in waves:
                wavefront.scale_to_fit_width(wavefront.width + 0.1)
                wavefront.set_stroke(opacity = 1 - wavefront.width/20)
                if (1 - wavefront.width/20) == 0:
                    waves.remove(wavefront)
    

        waves.add_updater(addwavefront)
        waves.add_updater(expandwaves)

        self.add(source)
        self.add(observer)
        self.add(waves)

        self.play(
                source.animate.shift(RIGHT*5), rate_func=linear, run_time=8
        )

        waves.remove_updater(addwavefront)

Manim Community v0.20.1

In [10]:
%%manim -qh DopplerSlide

class DopplerSlide(Scene):
    def construct(self):
        earth = Circle(radius=10, color=BLUE, fill_opacity=0.1)
        earth.move_to(DOWN * 12)
        
        # Set color to WHITE to invert the SVG for dark background
        observer = SVGMobject("ant.svg").scale(0.5).set_color(WHITE)
        observer.move_to(earth.get_top())
        obs_label = Text("Observer", color=WHITE, font_size=20).next_to(observer, DOWN, buff=0.2)
        
        orbit = Arc(
            radius=14, 
            angle=PI/3, 
            start_angle=PI/2 - PI/6, 
            arc_center=earth.get_center(), 
            color=DARK_GRAY
        )

        # Scale and shift the static background elements first
        main_group = Group(earth, observer, obs_label, orbit)
        main_group.scale(0.85).shift(UP * 1.5)

        satellite = ImageMobject("satellite.png").scale(0.1 * 0.85)
        satellite.rotate(PI/10)
        # Store current tracking angle to allow incremental absolute rotation 
        satellite.current_angle = 0
        sat_label = Text("", color=WHITE, font_size=16)
        
        rf_group = VGroup()

        p_tracker = ValueTracker(0.0)

        def update_satellite(mob):
            p = p_tracker.get_value()
            
            # Rotation tracking: Find proper tangent vector of orbit slightly around `p`  
            p1 = max(0, p - 0.001)
            p2 = min(1, p + 0.001)
            tangent_vec = orbit.point_from_proportion(p2) - orbit.point_from_proportion(p1)
            target_angle = angle_of_vector(tangent_vec)
            
            # Rotate by the delta to strictly enforce absolute rotation visually
            mob.rotate(target_angle - mob.current_angle)
            mob.current_angle = target_angle
            
            mob.move_to(orbit.point_from_proportion(p))

        satellite.add_updater(update_satellite)

        def update_labels_and_wave(mob):
            p = p_tracker.get_value()
            
            # Calculate mathematical position independently to prevent 1-frame lagging 
            sat_pos = orbit.point_from_proportion(p)

            # Smooth sigmoid frequency: goes smoothly from ~25 to ~8
            freq = 8 + 17 / (1 + np.exp(10 * (p - 0.5)))

            # Smooth color transition
            if p < 0.5:
                col = interpolate_color(BLUE, ORANGE, p * 2)
            else:
                col = interpolate_color(ORANGE, RED, (p - 0.5) * 2)

            if p < 0.4:
                lbl = "Approaching"
            elif p > 0.6:
                lbl = "Moving away"
            else:
                lbl = "Overhead"

            sat_label.become(Text(lbl, color=WHITE, font_size=16).next_to(satellite, UP, buff=0.1))

            link_line = Line(observer.get_center(), sat_pos)
            # derive phase as a reverse sigmoid to match the frequency curve, scaled for visual effect
            phase = 60 * ( 1 / (1 + np.exp(10 * (p - 0.5)+1)))

            new_wave = FunctionGraph(
                lambda x: 0.1 * np.sin(freq * x - phase), 
                x_range=[0, max(0.01, link_line.get_length())],
                color=col
            )

            # Fix for oscillation/jitter: 
            # We align using ORIGIN (which anchors the mathematical axis [0,0,0] of the 
            # FunctionGraph) rather than .get_start()
            new_wave.rotate(link_line.get_angle(), about_point=ORIGIN)
            new_wave.shift(observer.get_center())
            
            mob.become(new_wave)

        rf_group.add_updater(update_labels_and_wave)

        self.add(earth, obs_label, orbit, sat_label, rf_group, satellite, observer)
        self.play(p_tracker.animate.set_value(1.0), run_time=10, rate_func=linear)

Manim Community v0.20.1